In [1]:
# Import required libraries
import os
import pandas as pd
from utils import log_standard_scale, peek
from pathlib import Path
RAW_DIR = Path("../data")

In [2]:
# Load the dataset
df = pd.read_csv('../data/train_transcriptomics_SDY2867.tsv', sep='\t')
df = df.drop(columns=['transcriptomics_id'])
df.head()

,participant_id,timepoint,ensembl_gene_id,raw_count,tpm_count,material
0,SDY2867.SUB389725,0,ENSG00000000003,2.0,0.038336,Whole blood
1,SDY2867.SUB389727,-14,ENSG00000000003,10.0,0.200864,Whole blood
2,SDY2867.SUB389692,1,ENSG00000000003,16.0,0.964728,Whole blood
3,SDY2867.SUB389670,28,ENSG00000000003,17.0,0.670581,Whole blood
4,SDY2867.SUB389681,0,ENSG00000000003,2.0,0.094223,Whole blood


In [3]:
# raw_count has actual values (no nulls); material is uniformly 'Whole blood'
# SDY2867 has 5 timepoints: [-14, 0, 1, 7, 28] vs 2 for 2020_UGA
# Drop material before pivoting; drop raw_count to keep tpm_count only, consistent with 2019/2020 pipeline
print('raw_count null fraction:', df['raw_count'].isna().mean())
print('material unique values:', df['material'].unique())
print('timepoints:', sorted(df['timepoint'].unique()))
print('participants:', df['participant_id'].nunique())

raw_count null fraction: 0.0
material unique values: <ArrowStringArray>
['Whole blood']
Length: 1, dtype: str
timepoints: [np.int64(-14), np.int64(0), np.int64(1), np.int64(7), np.int64(28)]
participants: 73


In [4]:
# Drop material (entirely 'Whole blood'); drop raw_count to match pipeline (tpm_count only)
df = df.drop(columns=['raw_count', 'material'])
df.head()

,participant_id,timepoint,ensembl_gene_id,tpm_count
0,SDY2867.SUB389725,0,ENSG00000000003,0.038336
1,SDY2867.SUB389727,-14,ENSG00000000003,0.200864
2,SDY2867.SUB389692,1,ENSG00000000003,0.964728
3,SDY2867.SUB389670,28,ENSG00000000003,0.670581
4,SDY2867.SUB389681,0,ENSG00000000003,0.094223


In [5]:
unique_genes_challenge = set(
    pd.read_csv(RAW_DIR / "challenge_transcriptomics.tsv", sep='\t')['ensembl_gene_id'].unique()
)
len(unique_genes_challenge)

54902

In [6]:
df['timepoint'].unique()

array([  0, -14,   1,  28,   7])

In [7]:
# Filter to only timepoints 0, 7. Timepoints -14 and 1 not in the challenge set
df = df[df['timepoint'].isin([0, 7])]

# Filter to only challenge genes
df = df[df["ensembl_gene_id"].isin(unique_genes_challenge)]

# Pivot: each row is a participant_id, each gene+timepoint becomes a column
df_pivot = df.pivot_table(
    index='participant_id',
    columns=['timepoint', 'ensembl_gene_id'],
    values='tpm_count'
)

# Flatten multi-level columns to "d{timepoint}_{gene}" format
df_pivot.columns = [f'd{int(tp)}_{gene}' for tp, gene in df_pivot.columns]
df_pivot = df_pivot.reset_index()

print(df_pivot.shape)

(73, 109805)


In [8]:
peek(df_pivot)

,participant_id,d0_ENSG00000000003,d0_ENSG00000000005,d0_ENSG00000000419,d0_ENSG00000000457,d0_ENSG00000000460,d0_ENSG00000000938,d0_ENSG00000000971,d0_ENSG00000001036,d0_ENSG00000001084,d0_ENSG00000001167,d0_ENSG00000001460,d0_ENSG00000001461,d0_ENSG00000001497,d0_ENSG00000001561,d0_ENSG00000001617,d0_ENSG00000001626,d0_ENSG00000001629,d0_ENSG00000001630,d0_ENSG00000001631
0,SDY2867.SUB389659,0.375210,0.0,37.867705,8.410112,1.897157,452.649272,1.293022,12.468978,16.280764,25.572060,1.584164,20.541004,22.377562,2.935201,0.000000,0.0,11.267693,5.274519,16.350293
1,SDY2867.SUB389660,0.352071,0.0,39.929812,7.191947,2.261124,625.847485,2.344863,14.483491,17.114612,26.922821,1.251471,15.119491,19.991440,2.597309,0.028575,0.0,11.486776,3.815372,21.199825
2,SDY2867.SUB389661,0.180204,0.0,40.487763,9.215375,2.537860,374.571531,1.336281,14.495060,14.781612,22.180589,1.603173,23.022818,17.417415,3.091244,0.000000,0.0,15.421221,4.646479,18.984943
3,SDY2867.SUB389662,0.023781,0.0,38.825291,7.442696,2.001658,498.598388,2.780770,13.460020,10.641102,22.579521,1.416424,12.881570,19.319830,1.757225,0.000000,0.0,13.274277,5.076478,18.935624
4,SDY2867.SUB389663,0.101149,0.0,36.515188,7.454964,1.742845,445.988759,1.763300,11.588192,13.585535,21.708647,1.803702,16.690100,17.182578,2.721713,0.000000,0.0,10.101565,3.964353,16.373523


In [ ]:
df_pivot = log_standard_scale(df_pivot)
peek(df_pivot)

In [ ]:
# Save the cleaned dataset to the cleaned_data folder
os.makedirs('../cleaned_data', exist_ok=True)
df_pivot.to_parquet('../cleaned_data/transcriptomics_SDY2867_cleaned.parquet', index=False)